# Qwen2-VL API Server for SatQuery Hackathon
    
This notebook launches a FastAPI server exposing Qwen2-VL over a public ngrok URL. It is designed to be called by the local SatQuery backend.
Specifically, it uses `AdaptLLM/remote-sensing-Qwen2-VL-2B-Instruct` which is specially adapted for remote sensing.


In [ ]:
!pip install -U fastapi uvicorn pyngrok qwen-vl-utils transformers torchvision pillow pydantic accelerate nest-asyncio

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

MODEL_NAME = "AdaptLLM/remote-sensing-Qwen2-VL-2B-Instruct"
print(f"Loading processor and model: {MODEL_NAME} onto GPU...")

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype="auto", device_map="auto"
)
model.eval()
print("Model loaded successfully on device:", model.device)

In [ ]:
import base64
from io import BytesIO
from PIL import Image
from qwen_vl_utils import process_vision_info

# Dummy base64 images for testing if you don't have real ones
img1 = Image.new('RGB', (100, 100), color = 'red')
img2 = Image.new('RGB', (100, 100), color = 'blue')
buf1, buf2 = BytesIO(), BytesIO()
img1.save(buf1, format="PNG")
img2.save(buf2, format="PNG")
img1_b64 = base64.b64encode(buf1.getvalue()).decode("utf-8")
img2_b64 = base64.b64encode(buf2.getvalue()).decode("utf-8")

def do_local_inference(query, images):
    messages = [{"role": "user", "content": []}]
    for b64_img in images:
        img_data = base64.b64decode(b64_img)
        img = Image.open(BytesIO(img_data)).convert("RGB")
        messages[0]["content"].append({"type": "image", "image": img})
    
    # Image must be before text for RS-Qwen
    messages[0]["content"].append({"type": "text", "text": query})
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)
    
    generated_ids = model.generate(**inputs, max_new_tokens=256)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output_text[0]

print("--- LOCAL SINGLE_IMAGE ---")
try:
    ans = do_local_inference(
        "What type of land cover is visible in this image? Describe only what can reasonably be inferred from the image.",
        [img1_b64]
    )
    print("Answer:", ans)
    print("STATUS: PASS")
    status_single = "PASS"
except Exception as e:
    print("STATUS: FAIL", str(e))
    status_single = "FAIL"

print("\n--- LOCAL BI_TEMPORAL ---")
try:
    ans = do_local_inference(
        "Image 1 is the earlier observation (T1). Image 2 is the later observation (T2). Compare the two observations and describe visible changes. Distinguish direct visual observations from uncertain interpretation. Do not invent quantitative measurements.",
        [img1_b64, img2_b64]
    )
    print("Answer:", ans)
    print("STATUS: PASS")
    status_bi = "PASS"
except Exception as e:
    print("STATUS: FAIL", str(e))
    status_bi = "FAIL"

print("\n--- LOCAL OPTICAL_SAR ---")
try:
    ans = do_local_inference(
        "The first image is optical imagery and the second image is SAR imagery. Compare the information visible in both modalities and describe what can reasonably be inferred. Distinguish observations from uncertainty.",
        [img1_b64, img2_b64]
    )
    print("Answer:", ans)
    print("STATUS: PASS")
    status_sar = "PASS"
except Exception as e:
    print("STATUS: FAIL", str(e))
    status_sar = "FAIL"


In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List

app = FastAPI()

class AnalyzeRequest(BaseModel):
    query: str
    images: List[str]
    task_type: str

@app.get("/health")
async def health():
    return {"status": "ok", "model": MODEL_NAME}

@app.post("/analyze")
async def analyze(req: AnalyzeRequest):
    try:
        messages = [{"role": "user", "content": []}]
        for b64_img in req.images:
            img_data = base64.b64decode(b64_img)
            img = Image.open(BytesIO(img_data)).convert("RGB")
            messages[0]["content"].append({"type": "image", "image": img})
        
        messages[0]["content"].append({"type": "text", "text": req.query})
        
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to(model.device)
        
        generated_ids = model.generate(**inputs, max_new_tokens=256)
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        
        return {"answer": output_text[0], "model": MODEL_NAME, "task_type": req.task_type}
    except Exception as e:
        print(f"Error during inference: {e}")
        raise HTTPException(status_code=500, detail=str(e))


In [ ]:
import nest_asyncio
import uvicorn
import threading
import time
from pyngrok import ngrok, conf

# Optional: Add ngrok token if you have one
import getpass
print("Enter ngrok auth token (optional, press Enter to skip):")
token = getpass.getpass()
if token.strip():
    ngrok.set_auth_token(token.strip())

# Explicitly tell pyngrok to use local bin if needed, but defaults usually work in Colab.
public_url = ngrok.connect(8000).public_url
print('='*60)
print(f'\n>>> COPY THIS TO YOUR TERMINAL <<<\n')
print(f'export COLAB_API_URL={public_url}\n')
print('='*60)

nest_asyncio.apply()
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000)

threading.Thread(target=run_server, daemon=True).start()
time.sleep(3) # Wait for server to boot
print('Server is running in the background!')


In [ ]:
import requests

print("--- LOCAL FASTAPI /health ---")
try:
    res = requests.get('http://localhost:8000/health', timeout=10)
    print(res.json())
    status_health_local = "PASS"
except Exception as e:
    print(e)
    status_health_local = "FAIL"

print("\n--- PUBLIC NGROK /health ---")
try:
    res = requests.get(f'{public_url}/health', timeout=10)
    print(res.json())
    status_health_public = "PASS"
except Exception as e:
    print(e)
    status_health_public = "FAIL"

print("\n--- PUBLIC /analyze ---")
try:
    payload = {
        'query': 'Describe the visual scene shown in the satellite imagery.',
        'images': [img1_b64],
        'task_type': 'SINGLE_IMAGE'
    }
    res = requests.post(f'{public_url}/analyze', json=payload, timeout=60)
    print(res.json())
    status_analyze = "PASS" if res.status_code == 200 else "FAIL"
except Exception as e:
    print(e)
    status_analyze = "FAIL"


In [ ]:
print("="*40)
print("FINAL VERIFICATION REPORT")
print("="*40)
print(f"MODEL:\n{MODEL_NAME}\n")
print(f"GPU:\n{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}\n")
print(f"LOCAL SINGLE_IMAGE:\n{status_single}\n")
print(f"LOCAL BI_TEMPORAL:\n{status_bi}\n")
print(f"LOCAL OPTICAL_SAR:\n{status_sar}\n")
print(f"FASTAPI /health:\n{status_health_local}\n")
print(f"PUBLIC NGROK /health:\n{status_health_public}\n")
print(f"PUBLIC /analyze:\n{status_analyze}\n")
print("ERRORS:\nNone (if all pass)")
print("="*40)
